In [1]:
%pip install psycopg2-binary

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Instala sqlalchemy si no está instalado
%pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Instala python-dotenv si no está instalado
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Usaremos como entrada:

../data/processed/ventas_validas.csv
../data/processed/ventas_invalidas.csv

cargaremos en PostgreSQL:

ventas_clean
ventas_error

INSTALL:
pip install pandas sqlalchemy psycopg2-binary python-dotenv

# Actividad 2.4 — Carga de datos a Base de Datos

## Objetivo

En este notebook implementaremos la etapa final del pipeline de datos: la carga controlada a una base de datos relacional PostgreSQL.

El flujo completo trabajado hasta ahora es:

1. Ingesta de datos
2. Limpieza y transformación
3. Validación estructural y semántica
4. Carga a base de datos ← etapa actual

En esta etapa cargaremos los registros válidos en la tabla `ventas_clean` y los registros rechazados en la tabla `ventas_error`.

## Importancia de la carga controlada

La carga de datos no consiste solo en insertar registros en una tabla. También implica:

- validar conexión a la base de datos
- verificar estructura de tablas
- respetar tipos de datos
- manejar errores
- registrar trazabilidad
- separar datos insertados y rechazados

Esto permite que el pipeline sea reproducible, auditable y confiable.

In [4]:
import pandas as pd
import os
import logging
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

print("Bibliotecas cargadas...")

Bibliotecas cargadas...


In [5]:
os.makedirs("../logs", exist_ok=True)

logging.basicConfig(
    filename="../logs/load_database.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Inicio del proceso de carga a base de datos")
print("Logger configurado correctamente")

Logger configurado correctamente


## Conexión a PostgreSQL

La conexión se realizará usando variables de entorno desde el archivo `.env`.

Esto evita dejar credenciales directamente escritas en el notebook.

DB_HOST=TU_HOST

DB_PORT=TU_PORT

DB_NAME=TU_NAME

DB_USER=TU_USUARIO

DB_PASSWORD=TU_PASSWORD

In [6]:
load_dotenv("../.env")

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(connection_string)

print("Cadena de conexión creada correctamente")

Cadena de conexión creada correctamente


In [7]:
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT current_database(), current_schema();"))
        print(result.fetchone())
        logging.info("Conexión a PostgreSQL exitosa")
except Exception as e:
    logging.error(f"Error de conexión a PostgreSQL: {e}")
    print("Error de conexión:", e)

('postgres', 'public')


## Creación/verificación de tablas

A continuación se crearán las tablas si no existen.

Usaremos:

- `ventas_clean`: registros válidos
- `ventas_error`: registros rechazados

In [8]:
sql_create_tables = """
CREATE TABLE IF NOT EXISTS fraude_clean (
  trans_num TEXT,
  trans_date_trans_time TEXT,
  cc_num NUMERIC,
  merchant TEXT,
  category TEXT,
  amt NUMERIC,
  gender TEXT,
  is_fraud INT,
  edad INT,
  estado_calidad TEXT,
  fuente TEXT
);

CREATE TABLE IF NOT EXISTS fraude_error (
  trans_num TEXT,
  trans_date_trans_time TEXT,
  cc_num TEXT,
  merchant TEXT,
  amt TEXT,
  gender TEXT,
  edad TEXT,
  estado_calidad TEXT,
  fuente TEXT,
  observacion TEXT
);
"""

try:
    with engine.begin() as conn:
        conn.execute(text(sql_create_tables))
    logging.info("Tablas verificadas/creadas correctamente")
    print("Tablas verificadas o creadas correctamente")
except Exception as e:
    logging.error(f"Error creando tablas: {e}")
    print("Error creando tablas:", e)

Tablas verificadas o creadas correctamente


## Lectura de archivos validados

Los archivos generados en la etapa anterior se encuentran en `data/processed`.

- `ventas_validas.csv`
- `ventas_invalidas.csv`

In [9]:
ruta_validos = "../data/processed/fraude_validas.csv"
ruta_invalidos = "../data/processed/fraude_invalidas.csv"

df_validos = pd.read_csv(ruta_validos)
df_invalidos = pd.read_csv(ruta_invalidos)

print("Registros válidos:", len(df_validos))
print("Registros inválidos:", len(df_invalidos))

Registros válidos: 549427
Registros inválidos: 6292


In [10]:
df_validos.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,val_gender,val_is_fraud,val_amt_semantica,val_edad,val_lat_rango,val_long_rango,registro_valido,observaciones,estado_calidad,fuente
0,0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,...,True,True,True,True,True,True,True,Registro válido,valido,python_notebook
1,1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,...,True,True,True,True,True,True,True,Registro válido,valido,python_notebook
2,2,2020-06-21 12:14:53,3598215285024754,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,...,True,True,True,True,True,True,True,Registro válido,valido,python_notebook
3,3,2020-06-21 12:15:15,3591919803438423,fraud_Haley Group,misc_pos,60.05,Brian,Williams,M,32941 Krystal Mill Apt. 552,...,True,True,True,True,True,True,True,Registro válido,valido,python_notebook
4,4,2020-06-21 12:15:17,3526826139003047,fraud_Johnston-Casper,travel,3.19,Nathan,Massey,M,5783 Evan Roads Apt. 465,...,True,True,True,True,True,True,True,Registro válido,valido,python_notebook


In [11]:
df_invalidos.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,val_is_fraud,val_amt_semantica,val_edad,val_lat_rango,val_long_rango,registro_valido,observaciones,estado_calidad,fuente,observacion
0,112,2020-06-21 12:48:14,4587657402165341815,"fraud_Prosacco, Kreiger and Kovacek",home,26.03,Amber,Lewis,F,6296 John Keys Suite 858,...,True,True,False,True,True,False,Edad fuera de rango (18-120),error,python_notebook,Edad fuera de rango (18-120)
1,142,2020-06-21 12:59:36,371009169330125,fraud_Effertz LLC,entertainment,173.09,Randall,Burgess,M,393 Carroll Route Suite 197,...,True,True,False,True,True,False,Edad fuera de rango (18-120),error,python_notebook,Edad fuera de rango (18-120)
2,208,2020-06-21 13:24:28,3557442674264531,fraud_Cartwright PLC,kids_pets,111.57,Angela,Edwards,F,486 Wilkins Pines,...,True,True,False,True,True,False,Edad fuera de rango (18-120),error,python_notebook,Edad fuera de rango (18-120)
3,272,2020-06-21 13:47:13,3576021480694169,"fraud_Reichert, Huels and Hoppe",shopping_net,9.80,Dawn,Gray,F,9486 Joel Common Suite 554,...,True,True,False,True,True,False,Edad fuera de rango (18-120),error,python_notebook,Edad fuera de rango (18-120)
4,432,2020-06-21 14:40:35,4587657402165341815,fraud_Streich Ltd,home,7.46,Amber,Lewis,F,6296 John Keys Suite 858,...,True,True,False,True,True,False,Edad fuera de rango (18-120),error,python_notebook,Edad fuera de rango (18-120)


## Preparación de columnas para carga


Antes de insertar, seleccionaremos solo las columnas que existen en cada tabla destino.

In [12]:
columnas_clean = [
    "trans_num", "trans_date_trans_time", "cc_num", "merchant", 
    "category", "amt", "gender", "is_fraud", "edad", 
    "estado_calidad", "fuente"
]

columnas_error = [
    "trans_num", "trans_date_trans_time", "cc_num", "merchant", 
    "amt", "gender", "edad", "estado_calidad", "fuente", "observacion"
]

df_validos_carga = df_validos[columnas_clean].copy()
df_invalidos_carga = df_invalidos[columnas_error].copy()

df_validos_carga.head()

,trans_num,trans_date_trans_time,cc_num,merchant,category,amt,gender,is_fraud,edad,estado_calidad,fuente
0,2da90c7d74bd46a0caf3777415b3ebd3,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,M,0,52,valido,python_notebook
1,324cc204407e99f51b0d6ca0055005e7,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,F,0,30,valido,python_notebook
2,c81755dbbbea9d5c77f094348a7579be,2020-06-21 12:14:53,3598215285024754,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,F,0,50,valido,python_notebook
3,2159175b9efe66dc301f149d3d5abf8c,2020-06-21 12:15:15,3591919803438423,fraud_Haley Group,misc_pos,60.05,M,0,33,valido,python_notebook
4,57ff021bd3f328f8738bb535c302a31b,2020-06-21 12:15:17,3526826139003047,fraud_Johnston-Casper,travel,3.19,M,0,65,valido,python_notebook


In [13]:
df_invalidos_carga.head()

,trans_num,trans_date_trans_time,cc_num,merchant,amt,gender,edad,estado_calidad,fuente,observacion
0,bfcb7bca0abb957f3bb3aedf0c53a448,2020-06-21 12:48:14,4587657402165341815,"fraud_Prosacco, Kreiger and Kovacek",26.03,F,16,error,python_notebook,Edad fuera de rango (18-120)
1,e558e093053328aba4af71a4b98a9a4e,2020-06-21 12:59:36,371009169330125,fraud_Effertz LLC,173.09,M,17,error,python_notebook,Edad fuera de rango (18-120)
2,141db8af987da5cf06bc853a20f51e6a,2020-06-21 13:24:28,3557442674264531,fraud_Cartwright PLC,111.57,F,16,error,python_notebook,Edad fuera de rango (18-120)
3,8c3f3b2b91f1a39b5c6e44b38fb3986e,2020-06-21 13:47:13,3576021480694169,"fraud_Reichert, Huels and Hoppe",9.80,F,16,error,python_notebook,Edad fuera de rango (18-120)
4,6e7e8df3b997f3c381a4b81db3487834,2020-06-21 14:40:35,4587657402165341815,fraud_Streich Ltd,7.46,F,16,error,python_notebook,Edad fuera de rango (18-120)


## Validación previa a la carga

Validaremos que no existan datos críticos vacíos antes de insertar.

In [14]:
def validar_dataframe_para_carga(df, columnas_obligatorias):
    errores = []

    for columna in columnas_obligatorias:
        if columna not in df.columns:
            errores.append(f"Falta la columna obligatoria: {columna}")
        elif df[columna].isnull().sum() > 0:
            errores.append(f"La columna {columna} contiene valores nulos")

    return errores


errores_validos = validar_dataframe_para_carga(
    df_validos_carga,
    ["trans_num", "trans_date_trans_time", "amt", "estado_calidad"]
)

errores_invalidos = validar_dataframe_para_carga(
    df_invalidos_carga,
    ["estado_calidad", "observacion"]
)

print("Errores en válidos:", errores_validos)
print("Errores en inválidos:", errores_invalidos)

Errores en válidos: []
Errores en inválidos: []


## Carga de registros válidos

Insertaremos los registros válidos en la tabla `ventas_clean`.

In [15]:
try:
    if not errores_validos:
        df_validos_carga.to_sql(
            "fraude_clean",
            engine,
            if_exists="append",
            index=False,
            chunksize=5000, 
            method='multi'
        )
        logging.info(f"Registros válidos insertados: {len(df_validos_carga)}")
        print("Registros válidos insertados correctamente")
    else:
        logging.error(f"No se insertaron válidos por errores: {errores_validos}")
        print("No se insertaron válidos por errores previos")
except Exception as e:
    logging.error(f"Error insertando registros válidos: {e}")
    print("Error insertando registros válidos:", e)

Registros válidos insertados correctamente


## Carga de registros rechazados

Insertaremos los registros inválidos en la tabla `ventas_error`.

In [16]:
try:
    if not errores_invalidos:
        df_invalidos_carga.to_sql(
            "fraude_error",
            engine,
            if_exists="append",
            index=False,
            chunksize=5000,
            method='multi'
        )
        logging.info(f"Registros inválidos insertados en tabla de error: {len(df_invalidos_carga)}")
        print("Registros inválidos insertados correctamente en fraude_error")
    else:
        logging.error(f"No se insertaron inválidos por errores: {errores_invalidos}")
        print("No se insertaron inválidos por errores previos")
except Exception as e:
    logging.error(f"Error insertando registros inválidos: {e}")
    print("Error insertando registros inválidos:", e)

Registros inválidos insertados correctamente en fraude_error


## Verificación de carga

Consultaremos la base de datos para verificar cuántos registros quedaron cargados en cada tabla.

In [17]:
with engine.connect() as conn:
    total_clean = conn.execute(text("SELECT COUNT(*) FROM fraude_clean;")).scalar()
    total_error = conn.execute(text("SELECT COUNT(*) FROM fraude_error;")).scalar()

print("Total registros en fraude_clean:", total_clean)
print("Total registros en fraude_error:", total_error)

Total registros en fraude_clean: 549427
Total registros en fraude_error: 6292


In [18]:
consulta = """
SELECT fuente, COUNT(*) AS total
FROM fraude_clean
GROUP BY fuente;
"""
pd.read_sql(consulta, engine)

,fuente,total
0,python_notebook,549427


In [19]:
consulta_error = """
SELECT observacion, COUNT(*) AS total
FROM fraude_error
GROUP BY observacion
ORDER BY total DESC;
"""
pd.read_sql(consulta_error, engine)

,observacion,total
0,Edad fuera de rango (18-120),6292


## Trazabilidad del proceso

El proceso dejó registro en:

`../logs/load_database.log`

Ese archivo permite revisar:
- inicio del proceso
- conexión exitosa o fallida
- cantidad de registros insertados
- errores detectados durante la carga

In [20]:
logging.info("Fin del proceso de carga a base de datos")
print("Proceso de carga finalizado")

Proceso de carga finalizado


## Reflexión final

Responde brevemente:

1. ¿Cómo se garantizó la trazabilidad del proceso de carga?
2. ¿Qué mecanismos permiten reproducir nuevamente este proceso?
3. ¿Qué errores podrían ocurrir durante una carga a base de datos?
4. ¿Por qué es importante separar registros válidos y rechazados?
5. ¿Cómo se relaciona esta etapa con la validación estructural y semántica?

## Conclusión

En este notebook implementamos la etapa de carga de datos a PostgreSQL, usando como entrada los registros previamente validados.

Esta etapa permite consolidar el pipeline, dejando los datos persistidos en una base relacional y disponibles para análisis, visualización o modelamiento posterior.